# Assignment 1: Implementing and Analyzing a Basic RL Algorithm

Due: Tuesday May 12, before midnight EST

elsayed elmandoua - 20596379

In [1]:
import numpy as np
from enum import IntEnum
from copy import deepcopy
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
import matplotlib.colors as mcolors


action definitions

In [2]:
class Action(IntEnum):
    up = 0
    right = 1
    down = 2
    left = 3

action_to_str = {
    Action.up : "up",
    Action.right : "right",
    Action.down : "down",
    Action.left : "left",
}

action_to_offset = {
    Action.up : (-1, 0),
    Action.right : (0, 1),
    Action.down : (1, 0),
    Action.left : (0, -1),
}

gridworld class

In [3]:
class GridWorld:

    def __init__(self, height, width, goal, goal_value=5.0, danger=[], danger_value=-5.0, blocked=[], noise=0.0):
        """
        Initialize the GridWorld environment.
        Creates a gridworld like MDP
         - height (int): Number of rows
         - width (int): Number of columns
         - goal (int): Index number of goal cell
         - goal_value (float): Reward given for goal cell
         - danger (list of int): Indices of cells marked as danger
         - danger_value (float): Reward given for danger cell
         - blocked (list of int): Indices of cells marked as blocked (can't enter)
         - noise (float): probability of resulting state not being what was expected
        """

        self._width = width
        self._height = height
        self._grid_values = [0 for _ in range(height * width)] # Initialize state values.
        self._goal_value = goal_value
        self._danger_value = danger_value
        self._goal_cell = goal
        self._danger_cells = danger
        self._blocked_cells = blocked
        self._noise = noise # Noise level in the environment.
        assert noise >= 0 and noise < 1 # Ensure valid noise value.
        self.create_next_values() # Initialize the next state values.


    def reset(self):
        """
        Reset the state values to their initial state.
        """
        self._grid_values = [0 for _ in range(self._height * self._width)]
        self.create_next_values()


    def _inbounds(self, state):
        """
        Check if a state index is within the grid boundaries.
        """
        return state >= 0 and state < self._width * self._height

    def _inbounds_rc(self, state_r, state_c):
        """
        Check if row and column indices are within the grid boundaries.
        """
        return state_r >= 0 and state_r < self._height and state_c >= 0 and state_c < self._width

    def _state_to_rc(self, state):
        """
        Convert a state index to row and column indices.
        """
        return state // self._width, state % self._width

    def _state_from_action(self, state, action):
        """
        returns the resulting state after taking action from state
        if the move goes out of bounds or into a blocked cell, the agent stays
        """
        #TO DO:
        r, c = self._state_to_rc(state)
        dr, dc = action_to_offset[action]
        nr, nc = r + dr, c + dc

        if not self._inbounds_rc(nr, nc):
            return state
        
        new_state = nr * self._width + nc
        
        if new_state in self._blocked_cells:
            return state
        
        return new_state

    def is_terminal(self, state):
        """
        Returns true if a state is terminal (goal, or danger)
        """
        #To Do:
        return state == self._goal_cell or state in self._danger_cells

    def get_states(self):
        """
        gets all non-terminal non-blocked states — where bellman updates happen
        """
        #TO DO:
        return [
        s for s in range(self._height * self._width)
        if s not in self._blocked_cells and not self.is_terminal(s)
        ]

    def get_actions(self, state):
        """
        returns a list of valid actions given the current state
        all 4 actions available everywhere; transition model handles walls
        """
        #TO DO:
        return list(Action) 


    def get_reward(self, state):
        """
        get the reward for being in the current state
        """
        assert self._inbounds(state)
        # Reward is non-zero for danger or goal
        #TO DO:
        if state == self._goal_cell:
            return self._goal_value
        elif state in self._danger_cells:
            return self._danger_value
        return 0.0


    def get_transitions(self, state, action):
        """
        get a list of transitions as a result of attempting the action in the current state
        each item in the list is a dictionary, containing the probability of reaching that state and the state itself
        stochastic transition model
        returns list of {state, prob} dicts
        intended direction: prob = 1 - noise
        other 3 directions: prob = noise / 3 each
        probabilities accumulate for states reachable via multiple actions
        """
        #TO DO:
        transitions = {}
        
        if self._noise == 0.0:
            next_state = self._state_from_action(state, action)
            transitions[next_state] = transitions.get(next_state, 0) + 1.0
        
        else:
            intended = self._state_from_action(state, action)
            transitions[intended] = transitions.get(intended, 0) + (1 - self._noise)
        
            other_actions = [a for a in Action if a != action]
            noise_per = self._noise / len(other_actions)
            for other_action in other_actions:
                ns = self._state_from_action(state, other_action)
                transitions[ns] = transitions.get(ns, 0) + noise_per

        return [{"state": s, "prob": p} for s, p in transitions.items()]


    def get_value(self, state):
        """
        Get the current value of the state
        """
        assert self._inbounds(state)
        return self._grid_values[state]

    def create_next_values(self):
        """
        creates a temporary storage for state value updating
        if this is not used, then asynchronous updating may result in unexpected results
        to use properly, run this at the start of each iteration
        snapshot current values; updates go here until set_next_values() commits them
        """
        #TO DO:
        self._next_values = list(self._grid_values)

    def set_next_values(self):
        """
        set the state values from the temporary copied values
        to use properly, run this at the end of each iteration
        commit the buffered updates, synchronous bellman sweep
        """
        # TO DO:
        self._grid_values = list(self._next_values)

    def set_value(self, state, value):
        """
        set the value of the state into the temporary copy
        this value will not update into main storage until self.set_next_values() is called
        write to the buffer (not live until set_next_values is called
        """
        assert self._inbounds(state)
        #TO DO:
        self._next_values[state] = value

    def solve_linear_system(self, discount_factor=1.0):
        """
        solve the gridworld using a system of linear equations
        :param discount_factor: The discount factor for future rewards
        solves V = R + γ * P * V  as a linear system Ax = b
        under a uniform random policy (prob 1/4 per action)
        only valid for deterministic environments (noise=0)
        """
        #To Do:
        states = self.get_states()
        n = len(states)
        state_to_idx = {s: i for i, s in enumerate(states)}

        A = np.zeros((n, n))
        b = np.zeros(n)

        for i, state in enumerate(states):
            actions = self.get_actions(state)
            num_actions = len(actions)
            
            A[i][i] = 1.0  
            
            for action in actions:
                transitions = self.get_transitions(state, action)
                for t in transitions:
                    ns, prob = t["state"], t["prob"]
                    weight = (1.0 / num_actions) * prob  
                    
                    b[i] += weight * self.get_reward(ns)
                    
                    if not self.is_terminal(ns) and ns in state_to_idx:
                        A[i][state_to_idx[ns]] -= discount_factor * weight

        x = np.linalg.solve(A, b)

        for i, state in enumerate(states):
            self._grid_values[state] = x[i]

        return self


    def __str__(self):
        """
        Pretty print the state values
        """
        out_str = ""
        for r in range(self._height):
            for c in range(self._width):
                cell = r * self._width + c
                if cell in self._blocked_cells:
                    out_str += "{:>6}".format("----")
                elif cell == self._goal_cell:
                    out_str += "{:>6}".format("GOAL")
                elif cell in self._danger_cells:
                    out_str += "{:>6.2f}".format(self._danger_value)
                else:
                    out_str += "{:>6.2f}".format(self._grid_values[cell])
                out_str += " "
            out_str += "\n"
        return out_str

initialize your gridWorld and solve the linear system

In [4]:
simple_gw = GridWorld(height=5, width=5, goal=14, danger=[2, 18, 21], blocked=[6, 7, 11, 12], noise=0.0)
values_grid = simple_gw.solve_linear_system(discount_factor=0.95)
print(values_grid)

 -2.69  -3.60  -5.00  -1.64  -0.00 
 -2.35   ----   ----  -0.00   1.64 
 -2.51   ----   ----  -0.00   GOAL 
 -3.19  -3.91  -4.10  -5.00  -0.57 
 -3.82  -5.00  -3.99  -3.45  -1.82 



solver 2a: value iteration

In [5]:
def value_iteration(gw, discount, tolerance=0.1):
    #TO DO
    """
    perform value iteration to compute optimal state values
    bellman optimality: V*(s) = max_a Σ_{s'} P(s'|s,a) [R(s') + γ V*(s')]
    
    Args:
        gw (GridWorld): The GridWorld environment to solve.
        discount (float): Discount factor for future rewards (typically 0.75 or 0.95).
        tolerance (float): Convergence threshold; iteration stops when max value change < tolerance.
    
    Returns:
        int: Number of iterations performed before convergence.
    """
    iteration = 0
    convergence_history = []
 
    while True:
        gw.create_next_values()
        delta = 0.0
 
        for state in gw.get_states():
            old_val = gw.get_value(state)
            best_value = float('-inf')
 
            for action in gw.get_actions(state):
                action_value = 0.0
                for t in gw.get_transitions(state, action):
                    ns, prob = t["state"], t["prob"]
                    r = gw.get_reward(ns)
                    future = 0.0 if gw.is_terminal(ns) else gw.get_value(ns)
                    action_value += prob * (r + discount * future)
 
                best_value = max(best_value, action_value)
 
            gw.set_value(state, best_value)
            delta = max(delta, abs(best_value - old_val))
 
        gw.set_next_values()
        iteration += 1
        convergence_history.append(delta)
 
        if delta < tolerance:
            break
 
    return iteration, convergence_history

In [6]:
# Initialize your GridWorld
simple_gw = GridWorld(height=5, width=5, goal=14, danger=[2, 18, 21], blocked=[6, 7, 11, 12], noise=0.0)
noisy_gw = GridWorld(height=5, width=5, goal=14, danger=[2, 18, 21], blocked=[6, 7, 11, 12], noise=0.2)
discount = 0.95
tolerance = 0.1

In [7]:
value_iteration(simple_gw, discount, 0.1)
print(simple_gw)

  3.15   2.99  -5.00   4.51   4.75 
  3.32   ----   ----   4.75   5.00 
  3.49   ----   ----   5.00   GOAL 
  3.68   3.87   4.07  -5.00   5.00 
  3.49  -5.00   4.29   4.51   4.75 



In [8]:
value_iteration(noisy_gw, discount, 0.1)
print(noisy_gw)

  0.42  -0.10  -5.00   3.60   4.51 
  0.56   ----   ----   4.49   4.88 
  0.65   ----   ----   4.22   GOAL 
  0.72   0.83   1.40  -5.00   4.17 
  0.23  -5.00   2.09   2.90   3.84 



solver 2b: policy iteration

In [9]:
def policy_iteration(gw, discount, tolerance=0.1):
    #TO DO
    """
    alternates between:
        1. policy evaluation  - iterate V under fixed policy until convergence
        2. policy improvement - greedy update of policy using current V
    stops when policy is stable (no changes in improvement step)

    perform policy iteration to compute optimal state values and policy.
    
    alternates between policy evaluation (computing state values for current policy) 
    and policy improvement (updating policy to greedy actions) until convergence.
    
    args:
        gw (GridWorld): the gridWorld environment to solve.
        discount (float): discount factor for future rewards (typically 0.75 or 0.95).
        tolerance (float): convergence threshold for value updates within policy evaluation.
    
    returns:
        tuple: (iterations: int, policy: dict) where policy maps state indices to Action objects.
    """
    states = gw.get_states()
    policy = {s: Action.up for s in states} 
    outer_iteration = 0
 
    while True:
        while True:
            gw.create_next_values()
            delta = 0.0
 
            for state in states:
                old_val = gw.get_value(state)
                action = policy[state]
                value = 0.0
                for t in gw.get_transitions(state, action):
                    ns, prob = t["state"], t["prob"]
                    r = gw.get_reward(ns)
                    future = 0.0 if gw.is_terminal(ns) else gw.get_value(ns)
                    value += prob * (r + discount * future)
 
                gw.set_value(state, value)
                delta = max(delta, abs(value - old_val))
 
            gw.set_next_values()
            if delta < tolerance:
                break
 
        policy_stable = True
        for state in states:
            old_action = policy[state]
            best_action, best_val = None, float('-inf')
 
            for action in gw.get_actions(state):
                val = 0.0
                for t in gw.get_transitions(state, action):
                    ns, prob = t["state"], t["prob"]
                    r = gw.get_reward(ns)
                    future = 0.0 if gw.is_terminal(ns) else gw.get_value(ns)
                    val += prob * (r + discount * future)
 
                if val > best_val:
                    best_val, best_action = val, action
 
            policy[state] = best_action
            if best_action != old_action:
                policy_stable = False
 
        outer_iteration += 1
        if policy_stable:
            break
 
    return outer_iteration, policy

visualization helpers

In [10]:
def plot_values(gw, title="state values"):
    """heatmap of state values with annotations."""
    grid = np.zeros((gw._height, gw._width))
    mask = np.zeros((gw._height, gw._width), dtype=bool)
 
    for r in range(gw._height):
        for c in range(gw._width):
            cell = r * gw._width + c
            if cell in gw._blocked_cells:
                mask[r, c] = True
            elif cell == gw._goal_cell:
                grid[r, c] = gw._goal_value
            elif cell in gw._danger_cells:
                grid[r, c] = gw._danger_value
            else:
                grid[r, c] = gw.get_value(cell)
 
    fig, ax = plt.subplots(figsize=(6, 6))
    cmap = plt.cm.RdYlGn
    im = ax.imshow(grid, cmap=cmap, vmin=-6, vmax=6)
 
    for r in range(gw._height):
        for c in range(gw._width):
            cell = r * gw._width + c
            if cell in gw._blocked_cells:
                ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color='black'))
            elif cell == gw._goal_cell:
                ax.text(c, r, "G\n+5.0", ha='center', va='center', fontsize=10, color='white', fontweight='bold')
            elif cell in gw._danger_cells:
                ax.text(c, r, f"F\n-5.0", ha='center', va='center', fontsize=10, color='white')
            else:
                ax.text(c, r, f"{gw.get_value(cell):.2f}", ha='center', va='center', fontsize=9)
 
    plt.colorbar(im, ax=ax)
    ax.set_title(title)
    ax.set_xticks(range(gw._width))
    ax.set_yticks(range(gw._height))
    plt.tight_layout()
    plt.savefig(f"/mnt/user-data/outputs/{title.replace(' ', '_')}.png", dpi=150)
    plt.show()

In [11]:
def plot_policy(gw, policy, title="optimal policy"):
    """arrow visualization of the extracted policy."""
    arrow_map = {
        Action.up: (0, -0.3),
        Action.down: (0, 0.3),
        Action.left: (-0.3, 0),
        Action.right: (0.3, 0),
    }
 
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(-0.5, gw._width - 0.5)
    ax.set_ylim(-0.5, gw._height - 0.5)
    ax.set_aspect('equal')
    ax.invert_yaxis()
 
    for r in range(gw._height):
        for c in range(gw._width):
            cell = r * gw._width + c
            if cell in gw._blocked_cells:
                ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color='black'))
            elif cell == gw._goal_cell:
                ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color='green', alpha=0.5))
                ax.text(c, r, "G", ha='center', va='center', fontsize=12, color='white', fontweight='bold')
            elif cell in gw._danger_cells:
                ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color='red', alpha=0.5))
                ax.text(c, r, "F", ha='center', va='center', fontsize=12, color='white')
            else:
                if cell in policy:
                    dx, dy = arrow_map[policy[cell]]
                    ax.annotate("", xy=(c+dx, r+dy), xytext=(c, r),
                                arrowprops=dict(arrowstyle="->", color='steelblue', lw=2))
 
    ax.set_xticks(range(gw._width))
    ax.set_yticks(range(gw._height))
    ax.grid(True, alpha=0.3)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(f"/mnt/user-data/outputs/{title.replace(' ', '_')}.png", dpi=150)
    plt.show()